In [0]:
from pyspark.sql.functions import lit , col , when , current_timestamp , max , count , sum 

In [0]:
bronze_table = "globalretail.bronze.products_raw"

silver_table = "globalretail.silver.dim_products"

hwm_table = "globalretail.validation.silver_watermarks"

hwm_last_loaded_at = spark.sql(f"SELECT last_successful_load_at from {hwm_table} where entity_name='product'").collect()[0][0]
print(hwm_last_loaded_at)


max_last_loaded_at = spark.sql(f" SELECT max(ingested_at) from {bronze_table} where ingested_at > '{hwm_last_loaded_at}'").collect()[0][0]

print(max_last_loaded_at)

bronze_product_df = spark.read.table(bronze_table)

new_row_df = bronze_product_df.filter((col("ingested_at") > lit(hwm_last_loaded_at)) & (col("ingested_at") <= lit(max_last_loaded_at)))

display(new_row_df)




In [0]:
new_row_df.createOrReplaceTempView("vw_product_from_bronze")

In [0]:
cleaned = spark.sql("""
                    
                    select * ,

                    UPPER(TRIM(category)) as category_std,
                    trim(subcategory) as subcategory_std,
                    trim(brand) as brand_std,
                    UPPER(trim(currency_code)) as currency_code_std,

                    CASE
                        WHEN UPPER(TRIM(currency_code)) NOT IN ('USD', 'EUR', 'GBP', 'INR') THEN 'INVALID_CURRENCY'
                        WHEN unit_price IS NULL OR unit_price <= 0 THEN 'INVALID_PRICE'
                    ELSE 'VALID'
                    
                    END AS dq_status
                    from vw_product_from_bronze
                    
                    """)
display(cleaned)

In [0]:
rejects = cleaned.filter(col("dq_status") != "VALID").select("product_id", "sku", col("dq_status").alias("reject_reason"), current_timestamp().alias("rejected_at"))
reject_count = rejects.count()
if reject_count > 0:
    rejects.write.mode("append").saveAsTable("globalretail.silver.products_rejects")
    print(f"Rejected {reject_count} records")
else:
    print("No records to reject")

In [0]:
valid_latest = cleaned.filter(col("dq_status")=="VALID").select(
    col("product_id").cast("bigint"),
    "sku",
    "product_name",
    col("category_std").alias("category"),
    col("subcategory_std").alias("subcategory"),
    col("brand_std").alias("brand"),
    col("unit_price").cast("decimal(18,2)").alias("unit_price"),
    col("unit_cost").cast("decimal(18,2)").alias("unit_cost"),
    col("currency_code_std").alias("currency_code"),
    col("reorder_point").cast("int"),
    col("updated_at").cast("timestamp").alias("src_updated_at"),

)
valid_latest.createOrReplaceTempView("vw_product_source")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_product_changes AS
SELECT
    s.*,
    CASE
        WHEN t.product_id IS NULL THEN 'NEW'
        ELSE 'CHANGED'
    END AS change_type
FROM vw_product_source s
LEFT JOIN  globalretail.silver.dim_products as t
    ON s.product_id = t.product_id
   AND t.is_current = true;



select * from vw_product_changes;


In [0]:
%sql
MERGE INTO globalretail.silver.dim_products t
USING (
    SELECT *
    FROM vw_product_changes
    WHERE change_type = 'CHANGED'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY src_updated_at DESC) = 1
) c
ON t.product_id = c.product_id
   AND t.is_current = true

WHEN MATCHED THEN
  UPDATE SET
    t.effective_end_ts = c.src_updated_at - INTERVAL 1 SECOND,
    t.is_current = false,
    t.updated_at = current_timestamp()

In [0]:
%sql
INSERT INTO globalretail.silver.dim_products
SELECT
    abs(xxhash64(concat(cast(product_id AS STRING), '||', cast(src_updated_at AS STRING)))) AS product_sk,
    product_id,
    sku,
    product_name,
    category,
    subcategory,
    brand,
    unit_price,
    unit_cost,
    currency_code,
    reorder_point,
    src_updated_at AS effective_start_ts,
    TIMESTAMP('9999-12-31 23:59:59') AS effective_end_ts,
    true AS is_current,
    'VALID' AS dq_status,
    current_timestamp() AS created_at,
    current_timestamp() AS updated_at
FROM vw_product_changes
WHERE change_type IN ('NEW', 'CHANGED')

In [0]:

if max_last_loaded_at is not None:
    spark.sql(f"""
    UPDATE globalretail.validation.silver_watermarks
    SET   last_successful_load_at =  TIMESTAMP('{max_last_loaded_at}')
    WHERE entity_name = 'product'
    """)
else:
    print("No new records to process — watermark not updated.")